# B03. Saving and Loading

TidalPy can save a world or a system in two formats:

- **TOML config** (`save_to_toml`): a human-readable recipe of the identity, bulk properties, and layer structure. Loading it re-runs the builder.
- **Binary snapshot** (`save_binary`, read back with `load_world` or `load_system`): a copy of the object's state, restored without re-running the builder.

This notebook round-trips a world and a system through both formats and checks what each preserves.

In [1]:
import tempfile
from pathlib import Path

from TidalPy.Structures import build_world, build_system, load_world, load_system

work_dir = Path(tempfile.mkdtemp(prefix="tidalpy_demo_"))
print("writing files to a temporary directory")

writing files to a temporary directory


## Saving a World as TOML

`save_to_toml` writes the config recipe, with every default filled in. `build_world` rebuilds the world from the file's path.

In the `[layers.mantle.radiogenics]` table, `concentrations` holds each isotope's element concentration (uranium, for both U238 and U235) and `mass_fracs` the isotope's fraction of that element, so the two uranium entries share one concentration.

In [2]:
earth = build_world("earth_simple")

toml_path = work_dir / "earth.toml"
earth.save_to_toml(toml_path)
print(toml_path.read_text())

earth_from_toml = build_world(toml_path)
print("rebuilt:", repr(earth_from_toml))
print()

run_time = False
if run_time:
    print("Time to build a world from a toml:")
    %timeit build_world(toml_path)

# =====================================================================================================================
#  TidalPy world configuration: Earth-Simple
#  TidalPy version: 0.8.0
#  SciPy version: 1.18.0
#  CyRK version: 0.19.3
# =====================================================================================================================
schema_version = "0.2.0"
name = "Earth-Simple"
type = "terrestrial"
radius_m = 6371000.0
mass_kg = 5.972e+24
albedo = 0.3
emissivity = 1.0
obliquity_rad = 0.0
spin_frequency_rad_s = 7.292e-5

[tides]
global_tidal_model = "rheology"
min_degree_l = 2
max_degree_l = 2
eccentricity_trunc_lvl = 10
eccentricity_exact_tolerance = 0.0001
obliquity_trunc_lvl = 0
layer_tidal_heating = true
love_method = "radial_solver"

[eos_solver]
solve_temperature = false

[layers.inner_core]
layer_index = 0
radius_outer_m = 1221500.0
use_tides = true
is_volume_fixed = true
state = "auto"
is_static = true
is_incompressible = false
temperature_k = 5500.0
us

## Saving a World as a Binary Snapshot

`save_binary` writes the world's state: its bulk properties, its layers with their materials and models, and its tide and spin models. `load_world` reads it back as a new world of the class that saved it.

In [3]:
binary_path = work_dir / "earth.tpyb"
earth.save_binary(binary_path)
print("binary size:", binary_path.stat().st_size, "bytes")

restored = load_world(binary_path)
print("restored:", repr(restored))
print("tide model:", repr(restored.tide_model))
print()

run_time = False
if run_time:
    print("Time to load a world from binary:")
    %timeit load_world(binary_path)

binary size: 5682 bytes
restored: TerrestrialWorld('Earth-Simple', radius_km=6371, mass_kg=5.9720e+24, num_layers=3)
tide model: RheologyTide('rheology')



## Inspecting and Copying a World's Live State

`get_config_dict` returns the world's current state as a dictionary: identity, bulk properties, tide settings, and every layer with its material and models, with every default filled in. It has the form of a world config (it is what `save_to_toml` wrote above), so `build_world` rebuilds the world from it in memory. For a copy in memory, `world.copy()` goes through the binary record without a file (`copy.deepcopy` and `pickle` use the same path).

In [4]:
state = earth.get_config_dict()
print("state keys:", list(state))
print("layers:", list(state["layers"]))

# Rebuild in memory from the state dictionary
clone = build_world(state)
print("clone:", repr(clone))

# A copy through the binary record
earth_copy = earth.copy()
print("copy:", repr(earth_copy))

state keys: ['schema_version', 'name', 'type', 'radius_m', 'mass_kg', 'albedo', 'emissivity', 'obliquity_rad', 'spin_frequency_rad_s', 'tides', 'eos_solver', 'layers']
layers: ['inner_core', 'outer_core', 'mantle']
clone: TerrestrialWorld('Earth-Simple', radius_km=6371, mass_kg=5.9720e+24, num_layers=3)
copy: TerrestrialWorld('Earth-Simple', radius_km=6371, mass_kg=5.9720e+24, num_layers=3)


## Saving and Loading Systems

A whole system (host, star, and orbiting worlds with their orbital elements) supports the same two formats. The binary snapshot restores each world as its concrete type and keeps the orbital elements and roles, so derived quantities such as insolation are unchanged.

In [5]:
system = build_system("sol_system")

system_toml = work_dir / "sol_system.toml"
system_binary = work_dir / "sol_system.tpyb"
system.save_to_toml(system_toml)
system.save_binary(system_binary)
print("system toml:", system_toml.stat().st_size, "bytes")
print("system binary:", system_binary.stat().st_size, "bytes")

loaded = load_system(system_binary)
print("name:", loaded.name)
print("worlds:", [(world.name, type(world).__name__) for world in loaded])
print("star:", loaded.star.name)
print(f"Earth insolation, original: {system.calc_insolation_flux('earth'):.2f} W/m^2")
print(f"Earth insolation, loaded: {loaded.calc_insolation_flux('earth'):.2f} W/m^2")

system toml: 681 bytes
system binary: 7338 bytes
name: Sol System
worlds: [('sun', 'StarWorld'), ('earth', 'TerrestrialWorld'), ('jupiter', 'GasGiantWorld')]
star: sun
Earth insolation, original: 1361.35 W/m^2
Earth insolation, loaded: 1361.35 W/m^2


## Choosing a Format

- Use TOML when a person needs to read or edit the definition, when it belongs in version control, or when sharing a world. To reproduce a result elsewhere, keep the global configuration in effect beside it (`TidalPy.save_config`, notebook B01).
- Use binary to snapshot and restore an object's state, e.g., to cache a built system between runs. For this world, loading the binary file is about three times faster than rebuilding from TOML, and both take well under a millisecond.

A binary file stores every layer with its material and physics models, and the world's tide model, tide settings, and spin model. It does not store derived results: after loading, re-run the equation-of-state solve before any calculation that needs the interior.

In [6]:
import shutil

shutil.rmtree(work_dir, ignore_errors=True)  # clean up the temporary files
print("removed the temporary directory")

removed the temporary directory
